# Phase 11 — Streamlit application

This notebook explains how the app connects its widgets to the frozen Phase 10 model. It does **not** train a model or evaluate the test set again. The actual interface is in `../app.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.app_logic import (
    EXPECTED_FEATURE_ORDER, build_input_record, models_for_brand,
    numeric_widget_spec, validate_app_contract,
)
from src.predict import PriceRecommendationEngine

## 1. Load and verify the frozen contract

The production loader verifies every artifact hash before it loads CatBoost. The app also checks that its 11 controls still match the saved feature order.

In [ ]:
engine = PriceRecommendationEngine()
validate_app_contract(engine.feature_order, engine.allowed_inputs)
print('Model version:', engine.metadata['model_version'])
print('Feature order:', list(engine.feature_order))
print('Contract verified:', tuple(engine.feature_order) == EXPECTED_FEATURE_ORDER)

## 2. Build the same input record as the app

Numeric controls start at training medians. Their help text shows training ranges only as context. The model dropdown is constrained by the selected brand.

In [ ]:
numeric_defaults = {
    name: numeric_widget_spec(rule)['value']
    for name, rule in engine.schema['numeric_features'].items()
}
brand = 'Maruti'
available_models = models_for_brand(engine.allowed_inputs, brand)

car = build_input_record({
    'brand': brand,
    'model': available_models[0],
    'vehicle_age': numeric_defaults['vehicle_age'],
    'km_driven': numeric_defaults['km_driven'],
    'seller_type': 'Individual',
    'fuel_type': 'Petrol',
    'transmission_type': 'Manual',
    'mileage': numeric_defaults['mileage'],
    'engine': numeric_defaults['engine'],
    'max_power': numeric_defaults['max_power'],
    'seats': numeric_defaults['seats'],
})
car

## 3. Preview one production recommendation

This synthetic UI record is not a held-out test row. It exercises the same validated inference call used by Streamlit.

In [ ]:
recommendation = engine.recommend(car)
{
    'fair_price': recommendation['recommended_price_display'],
    'range': (
        recommendation['range_lower_display'],
        recommendation['range_upper_display'],
    ),
    'confidence': recommendation['confidence'],
    'confidence_reason': recommendation['confidence_reason'],
    'top_driver': recommendation['top_shap_drivers'][0]['explanation'],
}

## 4. Start the interface

Open a terminal in the project root and run:

```bash
python -m streamlit run app.py
```

The estimate is based on historical listings, not guaranteed sale prices. The app keeps this disclaimer and the known luxury, rare-category, Low-confidence, and high-price cautions visible.